Data Preparation

In [79]:
import pandas as pd
import glob


#amazon_data = glob.glob(r'C:\Users\youss\Downloads\ML\Ass 2\sentiment labelled sentences\amazon_cells_labelled.txt')
#imdb_data = glob.glob(r'C:\Users\youss\Downloads\ML\Ass 2\sentiment labelled sentences\imdb_labelled.txt')
#yelp_data = glob.glob(r'C:\Users\youss\Downloads\ML\Ass 2\sentiment labelled sentences\yelp_labelled.txt')

amazon = pd.read_csv(r'C:\Users\youss\Downloads\ML\Ass 2\sentiment labelled sentences\amazon_cells_labelled.txt', sep='\t', header=None, names=['sentence', 'label'])
imdb = pd.read_csv(r'C:\Users\youss\Downloads\ML\Ass 2\sentiment labelled sentences\imdb_labelled.txt', sep='\t', header=None, names=['sentence', 'label'])
yelp = pd.read_csv(r'C:\Users\youss\Downloads\ML\Ass 2\sentiment labelled sentences\yelp_labelled.txt', sep='\t', header=None, names=['sentence', 'label'])

df = pd.concat([amazon, imdb, yelp], ignore_index=True)


print('all data shape:', df.shape)
print('amazon data shape:', amazon.shape)
print('imdb data shape:', imdb.shape)
print('yelp data shape:', yelp.shape)
print(df['label'].value_counts())
df.head(10)




all data shape: (2748, 2)
amazon data shape: (1000, 2)
imdb data shape: (748, 2)
yelp data shape: (1000, 2)
label
1    1386
0    1362
Name: count, dtype: int64


,sentence,label
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1
5,I have to jiggle the plug to get it to line up...,0
6,If you have several dozen or several hundred c...,0
7,If you are Razr owner...you must have this!,1
8,"Needless to say, I wasted my money.",0
9,What a waste of money and time!.,0


Text Pre-Processing

In [80]:
df = df.dropna()
df = df.drop_duplicates(subset='sentence', keep='first')
df['sentence'] = df['sentence'].str.lower()

import re
def clean_text(text):
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df['sentence'] = df['sentence'].apply(clean_text)

print('all data shape after cleaning:', df.shape)
#hello world

all data shape after cleaning: (2731, 2)


Train/Test Split & Stratification

In [81]:
from sklearn.model_selection import train_test_split

x = df['sentence']
y = df['label']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print('x_train shape:', x_train.shape)
print('x_test shape:', x_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)


x_train shape: (2184,)
x_test shape: (547,)
y_train shape: (2184,)
y_test shape: (547,)


Test Representation

In [82]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
#meh
#ya vectorizer, read the text data and we e3mly kamoo9 then transform it into a matrix
x_train_vectorized = vectorizer.fit_transform(x_train)
x_test_vectorized = vectorizer.transform(x_test)
y_train_vectorized = y_train.values
y_test_vectorized = y_test.values

print('x_train_vectorized shape:', x_train_vectorized.shape)
print('x_test_vectorized shape:', x_test_vectorized.shape)
print('y_train_vectorized shape:', y_train_vectorized.shape)
print('y_test_vectorized shape:', y_test_vectorized.shape)

x_train_vectorized shape: (2184, 4587)
x_test_vectorized shape: (547, 4587)
y_train_vectorized shape: (2184,)
y_test_vectorized shape: (547,)


Multi-Nomial Naive Bayes

In [83]:
from sklearn.naive_bayes import MultinomialNB

#alpha=1.0 laplace smoothing
#fit_prior=True means we want to learn probabilities from the data BY HIMSELFFF
#class_prior=None means we want to use the default class prior probabilities (which are calculated from the training data)

nb = MultinomialNB(alpha=1.0, fit_prior=True, class_prior=None)
nb.fit(x_train_vectorized, y_train_vectorized)

#ya pc, "python is excellent" is a sentence, naive system opens the 2 boxes he has 1 is negative and the other is positive, he looks at the words in the sentence and counts how many times each word appears in each box, then he multiplies the probabilities of each word in the sentence then he says whether its positive or negative.

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[1084.,1100.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.7 ,-0.69]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 4587)","[[ 6., 1., 0.,..., 1., 0., 1.], [12., 2., 2.,..., 0., 1., 0.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 4587)","[[-7.83,-9.09,-9.78,...,-9.09,-9.78,-9.09], [-7.21,-8.67,-8.67,...,-9.77,-9.08,-9.77]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,4587


Model Evaluation (Accuracy, Confusion Matrix, Precision, Recall, F1-Score)

In [84]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, recall_score, precision_score, f1_score

y_pred = nb.predict(x_test_vectorized)
accuracy = accuracy_score(y_test_vectorized, y_pred)
recall = recall_score(y_test_vectorized, y_pred, average='weighted')
precision = precision_score(y_test_vectorized, y_pred, average='weighted')
f1 = f1_score(y_test_vectorized, y_pred, average='weighted')

print('Accuracy:', accuracy)
print('Recall:', recall)
print('Precision:', precision)
print('F1 Score:', f1)
print('Classification Report:\n', classification_report(y_test_vectorized, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test_vectorized, y_pred))



Accuracy: 0.7952468007312614
Recall: 0.7952468007312614
Precision: 0.7953373285495019
F1 Score: 0.7952495380003086
Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.80      0.79       271
           1       0.80      0.79      0.80       276

    accuracy                           0.80       547
   macro avg       0.80      0.80      0.80       547
weighted avg       0.80      0.80      0.80       547

Confusion Matrix:
 [[217  54]
 [ 58 218]]


New Sentences & Classify Them

In [85]:
new_sentences = [
    "I love this Volkswagen passat, it's amazing!",
    "I Love Love Love it! This is the worst movie I've ever seen",
    "I hate black monkeys, they are so annoying and stupid",
    "I love this movie, it's so good!",
    "i hate loving you"
]

cleaned_new_sentences = [clean_text(sentence) for sentence in new_sentences]
new_sentences_vectorized = vectorizer.transform(cleaned_new_sentences)

predictions = nb.predict(new_sentences_vectorized)

for sentence, prediction in zip(new_sentences, predictions):
    sentiment = 'Positive' if prediction == 1 else 'Negative'
    print(f'Sentence: "{sentence}" - Predicted Sentiment: {sentiment}')
    print(probability := nb.predict_proba(new_sentences_vectorized))

Sentence: "I love this Volkswagen passat, it's amazing!" - Predicted Sentiment: Positive
[[0.02562978 0.97437022]
 [0.36514958 0.63485042]
 [0.98968798 0.01031202]
 [0.05820006 0.94179994]
 [0.87555991 0.12444009]]
Sentence: "I Love Love Love it! This is the worst movie I've ever seen" - Predicted Sentiment: Positive
[[0.02562978 0.97437022]
 [0.36514958 0.63485042]
 [0.98968798 0.01031202]
 [0.05820006 0.94179994]
 [0.87555991 0.12444009]]
Sentence: "I hate black monkeys, they are so annoying and stupid" - Predicted Sentiment: Negative
[[0.02562978 0.97437022]
 [0.36514958 0.63485042]
 [0.98968798 0.01031202]
 [0.05820006 0.94179994]
 [0.87555991 0.12444009]]
Sentence: "I love this movie, it's so good!" - Predicted Sentiment: Positive
[[0.02562978 0.97437022]
 [0.36514958 0.63485042]
 [0.98968798 0.01031202]
 [0.05820006 0.94179994]
 [0.87555991 0.12444009]]
Sentence: "i hate loving you" - Predicted Sentiment: Negative
[[0.02562978 0.97437022]
 [0.36514958 0.63485042]
 [0.98968798 0.0

FINAL RESULTS

## 📊 Model Evaluation & Results Report

### 1. Final Test Accuracy
* **Overall Accuracy:** **79.5%** (0.795)

---

### 2. Performance Comparison (Positive vs. Negative Classes)

| Class | Precision | Recall | F1-Score | Support |
| :--- | :---: | :---: | :---: | :---: |
| **Negative (0)** | 0.79 | 0.80 | 0.79 | 271 |
| **Positive (1)** | 0.80 | 0.79 | 0.80 | 276 |

---

### 3. Summary & Performance Insights
* **Balanced Performance:** The Naive Bayes model shows a very well-balanced performance across both classes with an overall accuracy of **~80%**.
* **Class 0 (Negative):** Achieved a slightly higher **Recall (0.80)**, meaning it successfully identifies negative sentiments very well.
* **Class 1 (Positive):** Achieved a slightly higher **Precision (0.80)**, ensuring reliable positive predictions.
* **Conclusion:** The F1-scores ($\approx 0.79 - 0.80$) confirm that the model generalizes robustly and effectively handles the sentiment analysis task without leaning unfairly toward either class.